- Esse notebook servirá para a exploração e design inicial dos dados que serão utilizados como base do projeto.

In [1]:
# definição da janela de tempo do projeto --> últimos 5 anos --> alta rotatividade inerente da NFL
# está dinâmico pra precisar mudar apenas essa constante 'ANO_ATUAL'
ANO_ATUAL = 2026

ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

print(ultimos_5_anos)

[2021, 2022, 2023, 2024, 2025]


In [2]:
import nflreadpy as nfl
import pandas as pd
import pyarrow

# requisição da tabela de jogos dos anos especificados
print("Baixando o calendário...")
df_jogos = nfl.load_schedules(ultimos_5_anos).to_pandas()

# quero apenas os jogos da temporada regular (game_type == 'REG'), pois os playoffs são um ambiente totalmente distinto nesse contexto
# além de que a inclusão dos jogos de playoffs iria fazer com que certos times tivessem bem mais jogos que outros
df_regular = df_jogos[df_jogos['game_type'] == 'REG']

# apenas as colunas essenciais para começar
colunas_interesse = [
    'season', 'week', 'home_team', 'away_team', 
    'home_score', 'away_score'
]

df_base = df_regular[colunas_interesse].copy()

# requisição das estatísticas da temporada para features alternativas
print("Baixando estatísticas semanais de jardas e turnovers da API (pode levar alguns segundos)...")
df_stats = nfl.load_player_stats(ultimos_5_anos).to_pandas()

print("Esqueleto dos jogos pronto! Linhas:", len(df_base))
print("Estatísticas carregadas com sucesso! Linhas:", len(df_stats))

Baixando o calendário...
Baixando estatísticas semanais de jardas e turnovers da API (pode levar alguns segundos)...
Esqueleto dos jogos pronto! Linhas: 1359
Estatísticas carregadas com sucesso! Linhas: 94848


- Aqui eu quero manipular as features pra criar outras, mas no formato atual não dava pra fazer isso, então tive que manipular o dataframe pra que conseguisse organizar ele por times.

- Cada linha virou duas, pra representar os dois times.

In [3]:
# Buscando as colunas que têm 'team', 'int' ou 'fum' no nome
colunas_time = [col for col in df_stats.columns if 'team' in col.lower()]
colunas_erros = [col for col in df_stats.columns if 'int' in col.lower() or 'fum' in col.lower()]

print("Possíveis nomes para o time:", colunas_time)
print("Possíveis nomes para os erros:", colunas_erros)

Possíveis nomes para o time: ['team', 'opponent_team', 'special_teams_tds']
Possíveis nomes para os erros: ['passing_interceptions', 'sack_fumbles', 'sack_fumbles_lost', 'rushing_fumbles', 'rushing_fumbles_lost', 'receiving_fumbles', 'receiving_fumbles_lost', 'def_fumbles_forced', 'def_interceptions', 'def_interception_yards', 'def_fumbles', 'fumble_recovery_own', 'fumble_recovery_yards_own', 'fumble_recovery_opp', 'fumble_recovery_yards_opp', 'fumble_recovery_tds', 'fumbles_forced_by_opp', 'fumbles_not_forced', 'fumbles_out_of_bounds', 'fumbles_total', 'fumbles_lost_total', 'fantasy_points', 'fantasy_points_ppr']


In [4]:
# Buscando as colunas que indicam o volume de jogadas (tentativas de passe e corridas)
colunas_jogadas = [col for col in df_stats.columns if 'att' in col.lower() or 'carries' in col.lower()]
print("Possíveis nomes para as jogadas:", colunas_jogadas)

Possíveis nomes para as jogadas: ['attempts', 'carries', 'def_2pt_atts', 'fg_att', 'pat_att', 'gwfg_att', 'pt_att']


In [5]:
import numpy as np

# agregação das estatísticas de jogadores 
# filtrando apenas as colunas vitais para o modelo
colunas_stats = [
    'season', 'week', 'team',
    'passing_yards', 'rushing_yards', # Volume ofensivo
    'attempts', 'carries',            # Volume de jogadas
    'passing_interceptions', 'sack_fumbles_lost', 'rushing_fumbles_lost', 'receiving_fumbles_lost' # Erros
]
df_stats_clean = df_stats[colunas_stats].copy()

# preenchendo valores vazios (NaN) com 0 (ex: se ninguém sofreu fumble, o valor é 0)
df_stats_clean = df_stats_clean.fillna(0)

# agrupando os dados somando a produção de todos os jogadores do time naquela semana
df_team_stats = df_stats_clean.groupby(['season', 'week', 'team']).sum().reset_index()

# criando as features Jardas Totais, Turnovers, e Jardas por Jogada
df_team_stats['total_yards'] = df_team_stats['passing_yards'] + df_team_stats['rushing_yards']

df_team_stats['turnovers'] = (
    df_team_stats['passing_interceptions'] + 
    df_team_stats['sack_fumbles_lost'] + 
    df_team_stats['rushing_fumbles_lost'] + 
    df_team_stats['receiving_fumbles_lost']
)

df_team_stats['total_plays'] = df_team_stats['attempts'] + df_team_stats['carries']
df_team_stats['yards_per_play'] = np.where(
    df_team_stats['total_plays'] > 0, 
    df_team_stats['total_yards'] / df_team_stats['total_plays'], 
    0
)


# removendo as colunas quebradas de passe/corrida, mantendo só os totais
df_team_stats = df_team_stats[['season', 'week', 'team', 'total_yards', 'yards_per_play', 'turnovers']]

# reestruturação

# um DataFrame para a perspectiva do Mandante
df_home = df_base.copy()
df_home['team'] = df_home['home_team']
df_home['opponent'] = df_home['away_team']
df_home['points_scored'] = df_home['home_score']
df_home['points_allowed'] = df_home['away_score']
df_home['is_home'] = 1

# um DataFrame para a perspectiva do Visitante
df_away = df_base.copy()
df_away['team'] = df_away['away_team']
df_away['opponent'] = df_away['home_team']
df_away['points_scored'] = df_away['away_score']
df_away['points_allowed'] = df_away['home_score']
df_away['is_home'] = 0

# junta os dois DataFrames 
df_teams = pd.concat([df_home, df_away])

# merge: injetando as jardas e turnovers no df_teams
df_teams = pd.merge(
    df_teams, 
    df_team_stats, 
    on=['season', 'week', 'team'], 
    how='left'
)

# limpeza das colunas antigas/redundantes
colunas_para_remover = ['home_team', 'away_team', 'home_score', 'away_score']
df_teams = df_teams.drop(columns=colunas_para_remover)

# ordena cronologicamente por temporada, semana e time
df_teams = df_teams.sort_values(by=['season', 'week', 'team']).reset_index(drop=True)

# verificação
print("Colunas atuais:", list(df_teams.columns))
display(df_teams.head())

Colunas atuais: ['season', 'week', 'team', 'opponent', 'points_scored', 'points_allowed', 'is_home', 'total_yards', 'yards_per_play', 'turnovers']


,season,week,team,opponent,points_scored,points_allowed,is_home,total_yards,yards_per_play,turnovers
0,2021,1,ARI,TEN,38,13,0,425,6.538462,1
1,2021,1,ATL,PHI,6,32,1,288,4.721311,0
2,2021,1,BAL,LV,27,33,0,424,6.625000,2
3,2021,1,BUF,PIT,16,23,1,387,5.092105,1
4,2021,1,CAR,NYJ,19,14,1,390,6.290323,1


- Aqui serão implementadas as features que eu pensei pra englobar os jogos da NFL de maneira quase total, talvez sejam adicionadas novas em versões futuras.

In [6]:
# ordena cronologicamente por temporada, semana e time antes de qualquer cálculo, por segurança
df_teams = df_teams.sort_values(by=['team', 'season', 'week']).reset_index(drop=True)

# copia o DataFrame para garantir que o original não será alterado por acidente
df_features = df_teams.copy()

# agrupa por time para os cálculos de momento recente
grupo_times = df_features.groupby('team')

# Definindo o span da Média Móvel Exponencial (EWMA)
span_val = 3

# --- PONTOS ---
# Média de Pontos Marcados (Ataque) - EWMA (Momento) e Temporada atual
df_features['offense_pts_last3'] = grupo_times['points_scored'].transform(
    lambda x: x.shift(1).ewm(span=span_val, adjust=False, min_periods=1).mean()
)
df_features['offense_pts_season'] = df_features.groupby(['team', 'season'])['points_scored'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# Média de Pontos Sofridos (Defesa) - EWMA (Momento) e Temporada atual
df_features['defense_pts_last3'] = grupo_times['points_allowed'].transform(
    lambda x: x.shift(1).ewm(span=span_val, adjust=False, min_periods=1).mean()
)
df_features['defense_pts_season'] = df_features.groupby(['team', 'season'])['points_allowed'].transform(
    lambda x: x.shift(1).expanding().mean()
)


# --- JARDAS TOTAIS ---
# Volume Ofensivo - EWMA (Momento) e Temporada atual
df_features['total_yards_last3'] = grupo_times['total_yards'].transform(
    lambda x: x.shift(1).ewm(span=span_val, adjust=False, min_periods=1).mean()
)
df_features['total_yards_season'] = df_features.groupby(['team', 'season'])['total_yards'].transform(
    lambda x: x.shift(1).expanding().mean()
)


# --- JARDAS POR JOGADA ---
# Eficiência Ofensiva - EWMA (Momento) e Temporada atual
df_features['yards_per_play_last3'] = grupo_times['yards_per_play'].transform(
    lambda x: x.shift(1).ewm(span=span_val, adjust=False, min_periods=1).mean()
)
df_features['yards_per_play_season'] = df_features.groupby(['team', 'season'])['yards_per_play'].transform(
    lambda x: x.shift(1).expanding().mean()
)


# --- TURNOVERS ---
# Tendência a Erros - EWMA (Momento) e Temporada atual
df_features['turnovers_last3'] = grupo_times['turnovers'].transform(
    lambda x: x.shift(1).ewm(span=span_val, adjust=False, min_periods=1).mean()
)
df_features['turnovers_season'] = df_features.groupby(['team', 'season'])['turnovers'].transform(
    lambda x: x.shift(1).expanding().mean()
)

print("Métricas (Pontos, Jardas e Turnovers) com EWMA calculadas com sucesso!")

Métricas (Pontos, Jardas e Turnovers) com EWMA calculadas com sucesso!


- Vamos olhar como ficou a linha do tempo de um time específico (ex: TB) no começo de 2021

In [7]:
colunas_conferencia = [
    'season', 'week', 'team', 
    'points_scored', 'offense_pts_last3', 'offense_pts_season',
    'defense_pts_last3', 'defense_pts_season',
    'total_yards', 'total_yards_last3',
    'yards_per_play', 'yards_per_play_last3',
    'turnovers', 'turnovers_last3'
]

print("Conferência do Tampa Bay Buccaneers (TB):")
print(df_features[df_features['team'] == 'TB'][colunas_conferencia].head(17))

Conferência do Tampa Bay Buccaneers (TB):
      season  week team  points_scored  offense_pts_last3  offense_pts_season  \
2463    2021     1   TB             31                NaN                 NaN   
2464    2021     2   TB             48          31.000000           31.000000   
2465    2021     3   TB             24          39.500000           39.500000   
2466    2021     4   TB             19          31.750000           34.333333   
2467    2021     5   TB             45          25.375000           30.500000   
2468    2021     6   TB             28          35.187500           33.400000   
2469    2021     7   TB             38          31.593750           32.500000   
2470    2021     8   TB             27          34.796875           33.285714   
2471    2021    10   TB             19          30.898438           32.500000   
2472    2021    11   TB             30          24.949219           31.000000   
2473    2021    12   TB             38          27.474609          

- Resolvendo os "NaN"

- A regra será: preencher os valores faltantes da Semana 1 de um ano com a média final da temporada anterior daquele time.

- Como 2021 é o ano mais antigo do nosso dataset, não tem os dados de 2020 para puxar. Portanto, decidi aceitar os NaN e descartar esses primeiros jogos depois. Como tem 5 anos de dados (1.360 jogos), perder os jogos das 3 primeiras semanas de 2021 (cerca de 48 jogos) não vai machucar o modelo.

In [8]:
# --- PASSO 4: Preencher os NaN com a Média Final do Ano Anterior ---

# calcula a Média Final de cada time em cada temporada
medias_finais = df_features.groupby(['team', 'season'])[['points_scored', 'points_allowed', 'total_yards', 'yards_per_play', 'turnovers']].mean().reset_index()

# RENOMEIA as colunas para não haver confusão
medias_finais = medias_finais.rename(columns={
    'points_scored': 'media_final_ataque',
    'points_allowed': 'media_final_defesa',
    'total_yards': 'media_final_yards',
    'yards_per_play': 'media_final_ypp',
    'turnovers': 'media_final_turnovers'
})

# função que vai olhar pro passado
def preencher_nan_com_ano_anterior(row):
    # se a linha já tem os dados (não é NaN), retorna ela intacta
    if pd.notna(row['offense_pts_season']):
        return row

    time_atual = row['team']
    ano_atual = row['season']
    ano_anterior = ano_atual - 1

    # busca a Média Final do ano passado no dataframe auxiliar
    historico_ano_anterior = medias_finais[
        (medias_finais['team'] == time_atual) & 
        (medias_finais['season'] == ano_anterior)
    ]

    # se achou dados do ano passado (para 2022, 2023, 2024, 2025)
    if not historico_ano_anterior.empty:
        # preenche com as médias históricas
        row['offense_pts_last3'] = historico_ano_anterior['media_final_ataque'].values[0]
        row['offense_pts_season'] = historico_ano_anterior['media_final_ataque'].values[0]
        row['defense_pts_last3'] = historico_ano_anterior['media_final_defesa'].values[0]
        row['defense_pts_season'] = historico_ano_anterior['media_final_defesa'].values[0]
        row['total_yards_last3'] = historico_ano_anterior['media_final_yards'].values[0]
        row['total_yards_season'] = historico_ano_anterior['media_final_yards'].values[0]
        row['yards_per_play_last3'] = historico_ano_anterior['media_final_ypp'].values[0]
        row['yards_per_play_season'] = historico_ano_anterior['media_final_ypp'].values[0]
        row['turnovers_last3'] = historico_ano_anterior['media_final_turnovers'].values[0]
        row['turnovers_season'] = historico_ano_anterior['media_final_turnovers'].values[0]
    
    # se NÃO achou (caso exclusivo de 2021), não faz nada. Deixa como NaN.
    
    return row

# aplica a correção
df_features = df_features.apply(preencher_nan_com_ano_anterior, axis=1)

In [9]:
# --- Conferência ---
print("Semana 1 de 2022 do TB (Deve puxar a média final de 2021):")
print(df_features[(df_features['team'] == 'TB') & (df_features['season'] == 2022) & (df_features['week'] == 1)][['season', 'week', 'team', 
    'points_scored', 'offense_pts_last3', 'offense_pts_season',
    'defense_pts_last3', 'defense_pts_season',
    'total_yards', 'total_yards_last3',
    'yards_per_play', 'yards_per_play_last3',
    'turnovers', 'turnovers_last3']])

Semana 1 de 2022 do TB (Deve puxar a média final de 2021):
      season  week team  points_scored  offense_pts_last3  offense_pts_season  \
2480    2022     1   TB             19          30.058824           30.058824   

      defense_pts_last3  defense_pts_season  total_yards  total_yards_last3  \
2480          20.764706           20.764706          364              415.0   

      yards_per_play  yards_per_play_last3  turnovers  turnovers_last3  
2480        6.066667              6.347211          1         1.117647  


- Aqui será feito um merge pra desfazer a divisão de cada linha em duas outras, que foi realizada anteriormente. Além de uma limpeza final também.

In [10]:
# separando todas as colunas de interesse (Pontos + Jardas + Eficiência + Turnovers)
colunas_para_merge = [
    'season', 'week', 'team',
    'offense_pts_last3', 'offense_pts_season',
    'defense_pts_last3', 'defense_pts_season',
    'total_yards_last3', 'total_yards_season',
    'yards_per_play_last3', 'yards_per_play_season',
    'turnovers_last3', 'turnovers_season'
]

df_features_limpo = df_features[colunas_para_merge].copy()

# renomeando as colunas para o MANDANTE (Home)
df_features_home = df_features_limpo.rename(columns={
    'team': 'home_team',
    'offense_pts_last3': 'home_offense_pts_last3',
    'offense_pts_season': 'home_offense_pts_season',
    'defense_pts_last3': 'home_defense_pts_last3',
    'defense_pts_season': 'home_defense_pts_season',
    'total_yards_last3': 'home_total_yards_last3',
    'total_yards_season': 'home_total_yards_season',
    'yards_per_play_last3': 'home_yards_per_play_last3',
    'yards_per_play_season': 'home_yards_per_play_season',
    'turnovers_last3': 'home_turnovers_last3',
    'turnovers_season': 'home_turnovers_season'
})

# renomeando as colunas para o VISITANTE (Away)
df_features_away = df_features_limpo.rename(columns={
    'team': 'away_team',
    'offense_pts_last3': 'away_offense_pts_last3',
    'offense_pts_season': 'away_offense_pts_season',
    'defense_pts_last3': 'away_defense_pts_last3',
    'defense_pts_season': 'away_defense_pts_season',
    'total_yards_last3': 'away_total_yards_last3',
    'total_yards_season': 'away_total_yards_season',
    'yards_per_play_last3': 'away_yards_per_play_last3',
    'yards_per_play_season': 'away_yards_per_play_season',
    'turnovers_last3': 'away_turnovers_last3',
    'turnovers_season': 'away_turnovers_season'
})

# merge na base original
df_model = pd.merge(df_base, df_features_home, on=['season', 'week', 'home_team'], how='left')
df_model = pd.merge(df_model, df_features_away, on=['season', 'week', 'away_team'], how='left')

# removendo empates, por serem outliers muito improvaveis, apenas adicionam ruído
df_model = df_model[df_model['home_score'] != df_model['away_score']]

# criando a variável alvo (Target)
df_model['target'] = (df_model['home_score'] > df_model['away_score']).astype(int)

# limpeza final: remove os jogos de 2021 que ficaram com NaN por falta de histórico anterior
df_model_ml = df_model.dropna().reset_index(drop=True)

print("Dataset final para Machine Learning pronto!")
print(f"Total de jogos válidos: {len(df_model_ml)}")
print("Colunas prontas para o modelo:", list(df_model_ml.columns))

Dataset final para Machine Learning pronto!
Total de jogos válidos: 1339
Colunas prontas para o modelo: ['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score', 'home_offense_pts_last3', 'home_offense_pts_season', 'home_defense_pts_last3', 'home_defense_pts_season', 'home_total_yards_last3', 'home_total_yards_season', 'home_yards_per_play_last3', 'home_yards_per_play_season', 'home_turnovers_last3', 'home_turnovers_season', 'away_offense_pts_last3', 'away_offense_pts_season', 'away_defense_pts_last3', 'away_defense_pts_season', 'away_total_yards_last3', 'away_total_yards_season', 'away_yards_per_play_last3', 'away_yards_per_play_season', 'away_turnovers_last3', 'away_turnovers_season', 'target']


In [11]:
# --- Conferência Final do Dataset ---
print(f"Total de jogos após limpeza: {len(df_model_ml)}")
print(df_model_ml[[
    'season', 'week', 'home_team', 'away_team', 
    'home_offense_pts_season', 'home_total_yards_last3', 'home_yards_per_play_last3',
    'away_defense_pts_season', 'away_turnovers_last3', 'target'
]].head(3))

Total de jogos após limpeza: 1339
   season  week home_team away_team  home_offense_pts_season  \
0    2021     2       WAS       NYG                     16.0   
1    2021     2       CAR        NO                     19.0   
2    2021     2       CHI       CIN                     14.0   

   home_total_yards_last3  home_yards_per_play_last3  away_defense_pts_season  \
0                   261.0                   5.437500                     27.0   
1                   390.0                   6.290323                      3.0   
2                   350.0                   5.303030                     24.0   

   away_turnovers_last3  target  
0                   1.0       1  
1                   0.0       1  
2                   0.0       1  


- Terminando a engenharia inicial dos dados, e salvando o dataframe final:

In [12]:
# salvando o dataframe final em um arquivo Parquet
# o index=False garante que o Pandas não crie uma coluna extra de IDs
caminho_arquivo = 'nfl_features_ml_2021_2025.parquet'

try:
    df_model_ml.to_parquet(caminho_arquivo, index=False)
    print(f"Sucesso! Dataset de Machine Learning salvo em: {caminho_arquivo}")
    print("Você já pode abrir o Notebook 2.")
except Exception as e:
    print(f"Erro ao salvar, vamos usar CSV como fallback. Erro: {e}")
    df_model_ml.to_csv('nfl_features_ml_2021_2025.csv', index=False)

Sucesso! Dataset de Machine Learning salvo em: nfl_features_ml_2021_2025.parquet
Você já pode abrir o Notebook 2.
